In [ ]:
import calliope

MODEL_YAML = '../../model_Araucania/model.yaml'
cluster_number = '15'
SAVE_PATH  = f'../Cluster_Output_Araucania/Agglomerative/Agglomerative_k{cluster_number}/weighted_percentile_timeseries/Araucania_Agglomerative_k{cluster_number}.nc'

# Enable detailed logging to see model building and solving progress
calliope.set_log_verbosity('INFO', include_solver_output=True)

model = calliope.Model(MODEL_YAML)
model.run()
model.to_netcdf(SAVE_PATH)
print(f"Saved to: {SAVE_PATH}")

# Old Runs - old tech/weird cell
# 3 - 2m 38.6s
# 6 - 6m 25.5s
# 9 - 13m 22.6s
# 12 - 28m 25.5s

# New Runs - new tech/cleaned representative cell
# 3 - 3m 37.7s
# 6 - 11m 10.8s
# 9 - 24m 5.1s
# 12 - 55m 45.2
# 15 - 28m 8.9s

[2026-05-06 19:07:17] INFO     Model: initialising
[2026-05-06 19:07:18] INFO     `ammonia_storage` at `region_-37_75_-73_0` has no constraint to explicitly connect `energy_cap` to `storage_cap`, consider defining a `energy_cap_per_storage_cap_min/max/equals` constraint
[2026-05-06 19:07:18] INFO     `compressed_hydrogen_storage` at `region_-37_75_-73_0` has no constraint to explicitly connect `energy_cap` to `storage_cap`, consider defining a `energy_cap_per_storage_cap_min/max/equals` constraint
[2026-05-06 19:07:18] INFO     `ammonia_storage` at `region_-38_0_-72_5` has no constraint to explicitly connect `energy_cap` to `storage_cap`, consider defining a `energy_cap_per_storage_cap_min/max/equals` constraint
[2026-05-06 19:07:18] INFO     `compressed_hydrogen_storage` at `region_-38_0_-72_5` has no constraint to explicitly connect `energy_cap` to `storage_cap`, consider defining a `energy_cap_per_storage_cap_min/max/equals` constraint
[2026-05-06 19:07:18] INFO     `ammonia_storage

In [4]:
import pandas as pd

# --- Total system cost ---
total_cost_k_dollar = float(model.results['cost'].sum())
print(f"Total system cost: {total_cost_k_dollar:.3f} k$  ({total_cost_k_dollar/1e3:.3f} M$)")

# --- Carrier production ---
carrier_prod = model.results['carrier_prod'].to_dataframe().reset_index()
idx_col = next(c for c in carrier_prod.columns if 'loc_tech' in c.lower() and c != 'carrier_prod')
carrier_prod['carrier'] = carrier_prod[idx_col].str.split('::').str[-1]
carrier_prod['tech']    = carrier_prod[idx_col].str.split('::').str[1]

# Power (MWh) — wind + solar output
power_MWh = float(carrier_prod.loc[carrier_prod['carrier'] == 'power', 'carrier_prod'].clip(lower=0).sum())
# H2 (MWh) — electrolyser output, for cross-check
h2_MWh    = float(carrier_prod.loc[carrier_prod['carrier'] == 'hydrogen', 'carrier_prod'].clip(lower=0).sum())

print(f"\nPower generated : {power_MWh:>12,.1f} MWh")
print(f"H2 produced     : {h2_MWh:>12,.1f} MWh  ({h2_MWh/39.4:,.0f} tonnes)")

# --- LCOE ---
if power_MWh > 0:
    lcoe = (total_cost_k_dollar * 1e3) / power_MWh
    print(f"\nLCOE : {lcoe:.2f} $/MWh")
else:
    print("\nNo power production — check carrier name in carrier_prod")

# --- LCOH (cross-check) ---
H2_HHV_MWH_PER_KG = 39.4 / 1000
if h2_MWh > 0:
    lcoh = (total_cost_k_dollar * 1e3) / (h2_MWh / H2_HHV_MWH_PER_KG)
    print(f"LCOH : {lcoh:.3f} $/kg H₂")

# --- Cost breakdown by technology ---
cost_df = model.results['cost'].to_dataframe().reset_index()
cost_df['tech'] = cost_df['loc_techs_cost'].str.split('::').str[-1]
breakdown = cost_df.groupby('tech')['cost'].sum().sort_values(ascending=False)
breakdown_M = breakdown / 1e3
print("\nCost breakdown (M$):")
print(breakdown_M[breakdown_M.abs() > 0].to_string())

Total system cost: 634542.245 k$  (634.542 M$)

Power generated : 11,835,451.8 MWh
H2 produced     : 18,495,822.8 MWh  (469,437 tonnes)

LCOE : 53.61 $/MWh
LCOH : 1.352 $/kg H₂

Cost breakdown (M$):
tech
solar_single_axis                         242.844422
onshore_wind                              135.697055
electrolyser                              114.934970
ammonia_ccgt                               48.479550
compressed_hydrogen_storage                39.459763
ammonia_storage                            33.417015
haber_and_air_separation                   11.670254
dc_transmission:region_-38_75_-71_75        3.022613
dc_transmission:region_-38_25_-72_25        1.742058
dc_transmission:region_-38_75_-71_25        1.143390
hydrogen_pipeline:region_-38_75_-71_75      0.702954
hydrogen_pipeline:region_-38_75_-72_75      0.322116
dc_transmission:region_-38_75_-72_75        0.279367
hydrogen_pipeline:region_-38_75_-71_25      0.246846
hydrogen_pipeline:region_-38_25_-72_25      0.218800
d